In [1]:
import torch
import torch.nn as nn
def strip_mapanything_model(model):
    """
    移除 MapAnything 模型中 dpt_feature_head 之后的所有层。
    """
    
    # 定义在 dpt_feature_head 之后需要移除的子模块名称列表
    # 这些名称完全对应你提供的架构打印输出中的一级属性名
    modules_to_remove = [
        "dpt_regressor_head",
        "dense_head",  # 这是一个包含 DPTFeature 和 Regressor 的 Sequential
        "pose_head",
        "scale_head",
        "dense_adaptor",
        "pose_adaptor",
        "scale_adaptor"
    ]
    
    print("正在移除 dpt_feature_head 之后的参数...")
    
    for module_name in modules_to_remove:
        if hasattr(model, module_name):
            # 使用 delattr 从模型实例中彻底删除该属性
            delattr(model, module_name)
            print(f" -> [已删除]: {module_name}")
        else:
            print(f" -> [未找到/已删除]: {module_name}")
            
    print("模型剪枝完成。")
    return model

In [2]:
# Optional config for better memory efficiency
import os
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# 强制离线模式，避免触发任何联网下载
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

# Required imports
import torch
from mapanything.models import MapAnything

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"

HF_REPO = "facebook/map-anything-apache-v1"
DINO_CKPT_NAME = "dinov2_vitl14_pretrain.pth"


def find_local_hf_snapshot(repo_id: str) -> Path:
    hub_cache = Path(
        os.environ.get(
            "HF_HUB_CACHE",
            Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface")) / "hub",
        )
    )
    repo_cache_dir = hub_cache / f"models--{repo_id.replace('/', '--')}"
    snapshot_root = repo_cache_dir / "snapshots"
    if not snapshot_root.exists():
        raise FileNotFoundError(
            f"未找到本地 Hugging Face 缓存: {snapshot_root}\n"
            "请先联网下载一次模型，或把 local_model_dir 改成你本地模型目录。"
        )

    snapshots = sorted(
        [p for p in snapshot_root.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for snapshot_dir in snapshots:
        if (snapshot_dir / "config.json").exists() and (
            (snapshot_dir / "model.safetensors").exists()
            or (snapshot_dir / "pytorch_model.bin").exists()
        ):
            return snapshot_dir

    raise FileNotFoundError(f"{snapshot_root} 下没有找到可用模型文件。")


def ensure_dinov2_cache_exists() -> tuple[Path, Path]:
    torch_home = Path(os.environ.get("TORCH_HOME", Path.home() / ".cache" / "torch"))
    hub_dir = torch_home / "hub"
    ckpt_dir = hub_dir / "checkpoints"

    repo_candidates = [
        hub_dir / "facebookresearch_dinov2_main",
        hub_dir / "facebookresearch_dinov2_master",
    ]
    repo_dir = next((p for p in repo_candidates if p.exists()), None)
    if repo_dir is None:
        raise FileNotFoundError(
            f"未找到 DINOv2 Torch Hub 缓存仓库: {repo_candidates}\n"
            "请先联网下载一次 DINOv2 仓库后再离线运行。"
        )

    ckpt_path = ckpt_dir / DINO_CKPT_NAME
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"未找到 DINOv2 本地权重: {ckpt_path}\n"
            "请先联网下载一次 DINOv2 权重后再离线运行。"
        )

    return repo_dir, ckpt_path


def patch_torch_hub_offline_for_dinov2(dinov2_repo_dir: Path) -> None:
    original_torch_hub_load = torch.hub.load

    def offline_torch_hub_load(repo_or_dir, model, *args, **kwargs):
        if repo_or_dir == "facebookresearch/dinov2":
            # 关键点：走本地 repo + source='local'，避免 torch.hub 访问 GitHub
            kwargs.pop("force_reload", None)
            return original_torch_hub_load(
                str(dinov2_repo_dir),
                model,
                *args,
                source="local",
                **kwargs,
            )
        return original_torch_hub_load(repo_or_dir, model, *args, **kwargs)

    torch.hub.load = offline_torch_hub_load


local_model_dir = find_local_hf_snapshot(HF_REPO)
dinov2_repo_dir, dinov2_ckpt_path = ensure_dinov2_cache_exists()
patch_torch_hub_offline_for_dinov2(dinov2_repo_dir)

print(f"离线加载模型目录: {local_model_dir}")
print(f"离线 DINOv2 repo: {dinov2_repo_dir}")
print(f"离线 DINOv2 权重: {dinov2_ckpt_path}")

# 纯离线加载：只从本地目录读取，不访问 Hugging Face Hub / GitHub
model = strip_mapanything_model(
    MapAnything.from_pretrained(str(local_model_dir), local_files_only=True)
).to(device)


离线加载模型目录: /root/.cache/huggingface/hub/models--facebook--map-anything-apache-v1/snapshots/59bcb2c461d5439939f91e5608877db5c58363eb
离线 DINOv2 repo: /root/.cache/torch/hub/facebookresearch_dinov2_main
离线 DINOv2 权重: /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth
Loading pretrained dinov2_vitl14 from torch hub
Loading weights from local directory
正在移除 dpt_feature_head 之后的参数...
 -> [已删除]: dpt_regressor_head
 -> [已删除]: dense_head
 -> [已删除]: pose_head
 -> [已删除]: scale_head
 -> [已删除]: dense_adaptor
 -> [已删除]: pose_adaptor
 -> [已删除]: scale_adaptor
模型剪枝完成。


In [3]:
from mapanything.utils.image import load_images
# Load and preprocess images from a folder or list of paths
images = "test_images/rgb"  # or ["path/to/img1.jpg", "path/to/img2.jpg", ...]
views = load_images(images)
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=True,  # Trades off speed for more views (up to 2000 views on 140 GB). Trade off is negligible - see profiling section
    minibatch_size=None,              # Minibatch size for memory-efficient inference (use 1 for smallest GPU memory consumption). Default is dynamic computation based on available GPU memory.
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    use_multiview_confidence=False,   # Enable multi-view depth consistency based confidence in place of learning-based one
)


In [ ]:
print(len(predictions))
print(predictions[0].shape)
#6
# torch.Size([1, 1024, 37, 37])


6
torch.Size([1, 1024, 37, 37])
